In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import json
from typing import Any, Dict, List, Tuple, Union, Optional
import re

import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMClassifier, LGBMRegressor

from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    brier_score_loss
)

import sys

sys.path.append(str(Path.cwd().parent))

import utils

# Jogos - Carregamento

In [2]:
diretorio = Path("../api_outputs/jogos")

dataframes = []

for arquivo_json in diretorio.glob("202*/*.json"):
    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    df_arquivo = pd.DataFrame(dados)
    dataframes.append(df_arquivo)

df_jogos = pd.concat(dataframes, ignore_index=True)
print(df_jogos.shape)
df_jogos.head()

(1440, 5)


,jogo_id,edicao,rodada,equipe_mandante_id,equipe_visitante_id
0,900001,2022,1,111,114
1,900002,2022,1,105,109
2,900003,2022,1,108,120
3,900004,2022,1,107,124
4,900005,2022,1,102,103


# Atletas - Carregamento

In [3]:
diretorio = Path("../api_outputs/atletas")

dataframes = []

for arquivo_json in diretorio.glob("*.json"):
    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    df_arquivo = pd.DataFrame([dados])
    dataframes.append(df_arquivo)

df_atletas = pd.concat(dataframes, ignore_index=True)
print(df_atletas.shape)
df_atletas.head()

(2031, 4)


,atleta_id,apelido,posicao_id,clube_id
0,11201,Geraldo Bittencourt,3,109
1,10943,Zeca Henriques,1,107
2,11651,Zeca Carvalho,2,104
3,10410,Zeca Siqueira,5,122
4,10040,Rafael Peixoto,5,120


# Confrontos - Carregamento

In [4]:
diretorio = Path("../api_outputs/confrontos")

dataframes = []

for arquivo_json in diretorio.glob("202*/*.json"):
    nome = arquivo_json.stem
    padrao = r"confrontos_temporada_(\d+)_rodada_(\d+)"

    match = re.search(padrao, nome)
    if not match:
        print(f"[AVISO] Nome de arquivo fora do padrão esperado, ignorado: {arquivo_json.name}")
        continue

    temporada = match.group(1)
    rodada = match.group(2)

    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    df_arquivo = pd.DataFrame(dados)
    df_arquivo["temporada"] = temporada
    df_arquivo["rodada"] = rodada

    dataframes.append(df_arquivo)

df_confrontos = pd.concat(dataframes, ignore_index=True)
df_confrontos['temporada'] = df_confrontos['temporada'].astype(int)
df_confrontos['rodada'] = df_confrontos['rodada'].astype(int)
print(df_confrontos.shape)
df_confrontos.head()

(2784, 6)


,equipe_id,adversario_id,equipe_media_pontos_conquistados,adversario_media_pontos_cedidos,temporada,rodada
0,101,114,46.25,40.98,2022,11
1,102,122,46.05,52.91,2022,11
2,103,115,51.42,53.97,2022,11
3,105,126,50.68,55.02,2022,11
4,107,121,61.44,42.91,2022,11


# Equipes - Carregamento

In [5]:
diretorio = Path("../api_outputs/equipes")

dataframes = []

for arquivo_json in diretorio.glob("*.json"):
    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    df_arquivo = pd.DataFrame(dados)
    dataframes.append(df_arquivo)

df_equipes = pd.concat(dataframes, ignore_index=True)
print(df_equipes.shape)
df_equipes.head()

(28, 3)


,equipe_id,nome,sigla
0,101,Clube 101,C101
1,102,Clube 102,C102
2,103,Clube 103,C103
3,104,Clube 104,C104
4,105,Clube 105,C105


# Escalações - Carregamento

In [6]:
diretorio = Path("../api_outputs/escalacoes")

registros = []
arquivos_vazios_ou_invalidos = []

for arquivo_json in diretorio.glob("*.json"):
    jogo_id = arquivo_json.stem.replace('jogos_', '').replace('.json', '')

    with open(arquivo_json, "r", encoding="utf-8") as f:
        dados = json.load(f)

    if not isinstance(dados, dict):
        arquivos_vazios_ou_invalidos.append(arquivo_json.name)
        continue

    for equipe_id, equipe in dados.items():
        if not isinstance(equipe, dict):
            continue

        titulares = equipe.get("titulares") or []
        reservas = equipe.get("reservas") or []

        for jogador in titulares:
            if not isinstance(jogador, dict) or "atleta_id" not in jogador:
                continue
            registros.append({
                "jogo_id": jogo_id,
                "equipe_id": equipe_id,
                "atleta_id": jogador["atleta_id"],
                "titular": True,
                "momento_substituido": (jogador.get("substituido") or {}).get("momento"),
                "momento_entrou": None,
            })

        for jogador in reservas:
            if not isinstance(jogador, dict) or "atleta_id" not in jogador:
                continue
            registros.append({
                "jogo_id": jogo_id,
                "equipe_id": equipe_id,
                "atleta_id": jogador["atleta_id"],
                "titular": False,
                "momento_substituido": None,
                "momento_entrou": (jogador.get("entrou") or {}).get("momento"),
            })

df_escalacoes = pd.DataFrame(registros)

print(f"Total de registros: {df_escalacoes.shape[0]}")
print(f"Arquivos sem escalação (raiz não-dict): {len(arquivos_vazios_ou_invalidos)}")

if arquivos_vazios_ou_invalidos:
    print(arquivos_vazios_ou_invalidos[:10], "..." if len(arquivos_vazios_ou_invalidos) > 10 else "")

print(df_escalacoes.shape)
ordem = ['atleta_id',  'equipe_id', 'jogo_id', 'titular', 'momento_substituido', 'momento_entrou']
df_escalacoes = df_escalacoes[ordem]
df_escalacoes['equipe_id'] = df_escalacoes['equipe_id'].astype(int)
df_escalacoes['jogo_id'] = df_escalacoes['jogo_id'].astype(int)
df_escalacoes.head()

Total de registros: 101650
Arquivos sem escalação (raiz não-dict): 63
['jogos_901483.json', 'jogos_901270.json', 'jogos_901118.json', 'jogos_900924.json', 'jogos_901266.json', 'jogos_901267.json', 'jogos_900821.json', 'jogos_900118.json', 'jogos_901115.json', 'jogos_900801.json'] ...
(101650, 6)


,atleta_id,equipe_id,jogo_id,titular,momento_substituido,momento_entrou
0,10114,108,901354,True,49min,NaN
1,10125,108,901354,True,NaN,NaN
2,10309,108,901354,True,70min,NaN
3,10333,108,901354,True,NaN,NaN
4,10562,108,901354,True,NaN,NaN


# Base gato-mestre - Carregamento

In [7]:
base_gm = pd.read_csv('../../material_apoio/base_case_gm.csv')
base_gm.head()

,atleta_id,apelido,ano,rodada_id,clube_id,posicao_id,status_pre,status_inicial,preco_num,variacao_num,...,FS,PS,GS,GC,CA,CV,FC,I,PP,PC
0,11633,Kaique Oliveira,2024,20,127,4,Nulo,reserva,1.98,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11798,Geraldo Antunes,2025,13,110,6,PROVÁVEL,0,9.39,0.36,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10470,Pedro Peixoto,2024,14,127,3,Nulo,titular,2.8,-1.02,...,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0
3,10661,Adriel Barbosa,2025,36,115,4,Nulo,reserva,1.87,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10218,Norberto Guimarães,2022,30,118,5,Nulo,reserva,5.55,1.34,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


# Join entre as bases

In [8]:
join1 = base_gm.merge(
    df_jogos,
    left_on="match_id",
    right_on="jogo_id",
    how="left",
    suffixes=("", "_jogos"),
)

join2 = join1.merge(
    df_confrontos,
    left_on=["clube_id", "opponent", "ano", "rodada_id"],
    right_on=["equipe_id", "adversario_id", "temporada", "rodada"],
    how="left",
    suffixes=("", "_confrontos"),
)

join3 = join2.merge(
    df_escalacoes,
    left_on=["atleta_id", "match_id"],
    right_on=["atleta_id", "jogo_id"],
    how="left",
    suffixes=("", "_escalacoes"),
)

dados = join3

dados['preco_num'] = dados['preco_num'].str.replace(',', '.').astype(float)
dados['equipe_media_pontos_conquistados_ausencia'] = dados.shape[0] * [0]
dados['adversario_media_pontos_cedidos_ausencia'] = dados.shape[0] * [0]
dados['minutos_jogados_ausencia'] = dados.shape[0] * [0]
dados['momento_entrou_ausencia'] = dados.shape[0] * [0]
dados['momento_substituido_ausencia'] = dados.shape[0] * [0]
dados['rodada_id_corrigido'] = dados.shape[0] * [0]

linhas_duplicadas = dados.duplicated(subset=["atleta_id", "match_id"]).sum()
linhas_esperadas = len(base_gm)
linhas_obtidas = len(dados)

print(f"Linhas em base_gm (entrada): {linhas_esperadas}")
print(f"Linhas em dados (saída): {linhas_obtidas}")
print(f"Linhas duplicadas em (atleta_id, match_id): {linhas_duplicadas}")

print("Colunas finais:", list(dados.columns))

Linhas em base_gm (entrada): 117469
Linhas em dados (saída): 117469
Linhas duplicadas em (atleta_id, match_id): 1856
Colunas finais: ['atleta_id', 'apelido', 'ano', 'rodada_id', 'clube_id', 'posicao_id', 'status_pre', 'status_inicial', 'preco_num', 'variacao_num', 'media_num', 'jogos_num', 'pontos_num', 'minutos_jogados', 'entrou_em_campo', 'home_dummy', 'opponent', 'match_id', 'G', 'A', 'SG', 'FF', 'FT', 'FD', 'DD', 'DP', 'DE', 'DS', 'FS', 'PS', 'GS', 'GC', 'CA', 'CV', 'FC', 'I', 'PP', 'PC', 'jogo_id', 'edicao', 'rodada', 'equipe_mandante_id', 'equipe_visitante_id', 'equipe_id', 'adversario_id', 'equipe_media_pontos_conquistados', 'adversario_media_pontos_cedidos', 'temporada', 'rodada_confrontos', 'equipe_id_escalacoes', 'jogo_id_escalacoes', 'titular', 'momento_substituido', 'momento_entrou', 'equipe_media_pontos_conquistados_ausencia', 'adversario_media_pontos_cedidos_ausencia', 'minutos_jogados_ausencia', 'momento_entrou_ausencia', 'momento_substituido_ausencia', 'rodada_id_corrig

In [9]:
dados.head()

,atleta_id,apelido,ano,rodada_id,clube_id,posicao_id,status_pre,status_inicial,preco_num,variacao_num,...,jogo_id_escalacoes,titular,momento_substituido,momento_entrou,equipe_media_pontos_conquistados_ausencia,adversario_media_pontos_cedidos_ausencia,minutos_jogados_ausencia,momento_entrou_ausencia,momento_substituido_ausencia,rodada_id_corrigido
0,11633,Kaique Oliveira,2024,20,127,4,Nulo,reserva,1.98,0.00,...,900849.0,False,NaN,NaN,0,0,0,0,0,0
1,11798,Geraldo Antunes,2025,13,110,6,PROVÁVEL,0,9.39,0.36,...,901223.0,False,NaN,0min,0,0,0,0,0,0
2,10470,Pedro Peixoto,2024,14,127,3,Nulo,titular,2.80,-1.02,...,901049.0,True,89min,NaN,0,0,0,0,0,0
3,10661,Adriel Barbosa,2025,36,115,4,Nulo,reserva,1.87,0.00,...,901290.0,False,NaN,NaN,0,0,0,0,0,0
4,10218,Norberto Guimarães,2022,30,118,5,Nulo,reserva,5.55,1.34,...,900097.0,False,NaN,51min,0,0,0,0,0,0


# Verificação e tratamento de tipagem, valores ausentes e extremos

In [10]:
utils.data_exploratory_analysis(dados)

===== Data Inspection =====

Shape:
(117469, 60)

Data Types:
atleta_id                                      int64
apelido                                          str
ano                                            int64
rodada_id                                      int64
clube_id                                       int64
posicao_id                                     int64
status_pre                                       str
status_inicial                                   str
preco_num                                    float64
variacao_num                                 float64
media_num                                    float64
jogos_num                                      int64
pontos_num                                   float64
minutos_jogados                              float64
entrou_em_campo                                 bool
home_dummy                                   float64
opponent                                     float64
match_id                             

In [11]:
# ============================================================
# LIMPEZA E TRATAMENTO DE VALORES AUSENTES
# ============================================================
colunas = [
    'adversario_id', # Alta quantidade de valores ausentes
    'edicao', # Cerca de 5% de valores ausentes e coluna 'ano' representa a mesma informação
    'equipe_id', # Alta quantidade de valores ausentes
    'equipe_id_escalacoes', # Alta quantidade de valores ausentes
    'home_dummy', # Alta quantidade de valores ausentes
    'jogo_id_escalacoes', # Alta quantidade de valores ausentes
    'opponent', # Alta quantidade de valores ausentes e a coluna 'equipe_visitante_id' representa a mesma informação
    'rodada', # Cerca de 5% de valores ausentes
    'rodada_confrontos', # Alta quantidade de valores ausentes
    'temporada', # Alta quantidade de valores ausentes e coluna 'ano' representa a mesma informação
    'titular', # Alta quantidade de valores ausentes
    'jogo_id' # Cerca de 5% de valores ausentes
]

dados = dados.drop(columns=colunas).copy()

# Como as colunas 'equipe_mandante_id' e 'equipe_visitante_id' possuem
# aproximadamente 5% de valores ausentes, tais registros foram excluídos
dados = dados.dropna(subset=['equipe_mandante_id', 'equipe_visitante_id']).copy()

# Tratamento de valores ausentes
# preco_num
dados["preco_num"] = dados["preco_num"].fillna(
    dados.groupby("atleta_id")["preco_num"].transform("mean")
)

# minutos_jogados
dados.loc[
    (dados['minutos_jogados'].isna()) & (dados['entrou_em_campo'] == True),
    'minutos_jogados'
] = 90.0

dados.loc[
    (dados['minutos_jogados'].isna()) & (dados['entrou_em_campo'] != True),
    'minutos_jogados'
] = 0.0

dados.loc[
    (dados['status_inicial'] == '0'),
    'status_inicial'
] = 'reserva'

# momento_entrou
dados.loc[
    (dados['momento_entrou'].isna()) & (dados['entrou_em_campo'] == True),
    'momento_entrou'
] = "0min"

dados.loc[
    (dados['momento_entrou'].isna()) & (dados['entrou_em_campo'] != True),
    'momento_entrou'
] = "0min"

dados['momento_entrou'] = dados['momento_entrou'].str.replace('min', '').astype(int)

# momento_substituido
dados.loc[
    (dados['momento_substituido'].isna()) & (dados['entrou_em_campo'] == True),
    ['momento_substituido', 'momento_substituido_ausencia']
] = ["0min", 0]

dados.loc[
    (dados['momento_substituido'].isna()) & (dados['entrou_em_campo'] != True),
    ['momento_substituido', 'momento_substituido_ausencia']
] = ["0min", 1]

dados['momento_substituido'] = dados['momento_substituido'].str.replace('min', '').astype(int)

# equipe_media_pontos_conquistados
dados.loc[
    (dados['rodada_id'] == 1) & (dados['equipe_media_pontos_conquistados'].isna()),
    ['equipe_media_pontos_conquistados', 'equipe_media_pontos_conquistados_ausencia']
] = [0.0, 0]

dados.loc[
    (dados['rodada_id'] != 1) & (dados['equipe_media_pontos_conquistados'].isna()),
    ['equipe_media_pontos_conquistados', 'equipe_media_pontos_conquistados_ausencia']
] = [0.0, 1]

# adversario_media_pontos_cedidos
dados.loc[
    (dados['rodada_id'] == 1) & (dados['adversario_media_pontos_cedidos'].isna()),
    ['adversario_media_pontos_cedidos', 'adversario_media_pontos_cedidos_ausencia']
] = [0.0, 0]

dados.loc[
    (dados['rodada_id'] != 1) & (dados['adversario_media_pontos_cedidos'].isna()),
    ['adversario_media_pontos_cedidos', 'adversario_media_pontos_cedidos_ausencia']
] = [0.0, 1]

# status_inicial, apelido e DD
dados = dados.drop(columns=['status_inicial', 'apelido', 'DD'])



# ============================================================
# LIMPEZA E DUMMIES DA COLUNA 'status_pre'
# ============================================================
dados["status_pre"] = dados["status_pre"].str.strip().str.lower()
dados = pd.get_dummies(dados, columns=['status_pre'])



# ============================================================
# TRATAMENTO DE DUPLICATAS
# ============================================================
# Linhas duplicadas por completo
dados = dados.drop_duplicates(keep="first").copy()

# Linhas com as chaves 'atleta_id' e 'match_id' duplicadas
dados = (
    dados
    .sort_values("preco_num", ascending=False)
    .drop_duplicates(
        subset=["atleta_id", "match_id"],
        keep="first"
    )
).copy()



# ============================================================
# TRATAMENTO DE VALORES EXTREMOS
# ============================================================
def winsorizar_coluna(coluna, inferior=0.01, superior=0.99):
    limite_inferior = coluna.quantile(inferior)
    limite_superior = coluna.quantile(superior)

    return coluna.clip(
        lower=limite_inferior,
        upper=limite_superior
    )

# minutos_jogados
dados['minutos_jogados'] = winsorizar_coluna(dados['minutos_jogados'], 0.02, 0.98)

# rodada_id
dados.loc[
    (dados['rodada_id'] == 0),
    ['rodada_id', 'rodada_id_corrigido']
] = [1, 1]

dados.loc[
    (dados['rodada_id'] > 38),
    ['rodada_id', 'rodada_id_corrigido']
] = [38, 1]

# Feature engineering

In [12]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================

scouts_positivos = [
    "G",
    "A",
    "SG",
    "FF",
    "FT",
    "FD",
    "DP",
    "DE",
    "DS",
    "FS",
    "PS",
]

scouts_negativos = [
    "GS",
    "GC",
    "CA",
    "CV",
    "FC",
    "I",
    "PP",
    "PC",
]

scouts = scouts_positivos + scouts_negativos


# Variáveis adicionais para as quais serão construídas
# features históricas.
variaveis_historicas_adicionais = [
    "preco_num",
    "jogos_num",
    "pontos_num",
]


# Premissa adotada para posicao_id:
# 1 = goleiro
# 2 = lateral
# 3 = zagueiro
# 4 = meia
# 5 = atacante
# 6 = técnico

posicoes_defensivas = {1, 2, 3}
posicoes_ofensivas = {4, 5}
posicoes_atletas = posicoes_defensivas | posicoes_ofensivas


# ============================================================
# VALIDAÇÃO DAS COLUNAS
# ============================================================

colunas_obrigatorias = {
    "ano",
    "rodada_id",
    "rodada_id_corrigido",
    "match_id",
    "atleta_id",
    "clube_id",
    "equipe_mandante_id",
    "equipe_visitante_id",
    "posicao_id",
    "preco_num",
    "jogos_num",
    "pontos_num",
    "minutos_jogados",
    "status_pre_provável",
    *scouts,
}

colunas_ausentes = sorted(
    colunas_obrigatorias - set(dados.columns)
)

if colunas_ausentes:
    raise KeyError(
        "Colunas obrigatórias ausentes em 'dados': "
        f"{colunas_ausentes}"
    )


dados = dados.copy()


# ============================================================
# CONVERSÃO DOS TIPOS
# ============================================================

colunas_numericas = [
    *scouts,
    *variaveis_historicas_adicionais,
    "minutos_jogados",
    "status_pre_provável",
    "posicao_id",
]

for coluna in colunas_numericas:
    dados[coluna] = pd.to_numeric(
        dados[coluna],
        errors="coerce",
    )


# Atenção:
# Scouts ausentes serão tratados como zero.
# Isso pressupõe que NaN significa ausência da ação.
# Se NaN significar falha de coleta, remova este preenchimento.
dados[scouts] = dados[scouts].fillna(0)

# Mesma premissa para minutos jogados.
dados["minutos_jogados"] = (
    dados["minutos_jogados"].fillna(0)
)


# ============================================================
# VALIDAÇÃO DA GRANULARIDADE
# ============================================================

chave_atleta_partida = [
    "match_id",
    "atleta_id",
]

duplicados = dados.duplicated(
    subset=chave_atleta_partida,
    keep=False,
)

if duplicados.any():
    exemplos_duplicados = (
        dados.loc[
            duplicados,
            [
                "ano",
                "rodada_id",
                "match_id",
                "atleta_id",
                "clube_id",
            ],
        ]
        .sort_values(
            [
                "ano",
                "rodada_id",
                "match_id",
                "atleta_id",
            ]
        )
        .head(10)
    )

    raise ValueError(
        "Existem linhas duplicadas para atleta e partida.\n\n"
        f"Exemplos:\n{exemplos_duplicados}"
    )


# ============================================================
# ORDENAÇÃO TEMPORAL
# ============================================================

dados = (
    dados
    .sort_values(
        [
            "ano",
            "rodada_id",
            "match_id",
            "atleta_id",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# JOGO EM CASA E ADVERSÁRIO
# ============================================================

eh_mandante = dados["clube_id"].eq(
    dados["equipe_mandante_id"]
)

eh_visitante = dados["clube_id"].eq(
    dados["equipe_visitante_id"]
)

clube_fora_do_confronto = ~(
    eh_mandante | eh_visitante
)

if clube_fora_do_confronto.any():
    exemplos_invalidos = (
        dados.loc[
            clube_fora_do_confronto,
            [
                "ano",
                "rodada_id",
                "match_id",
                "atleta_id",
                "clube_id",
                "equipe_mandante_id",
                "equipe_visitante_id",
            ],
        ]
        .head(10)
    )

    raise ValueError(
        "Há atletas cujo clube_id não corresponde à equipe "
        "mandante nem à visitante.\n\n"
        f"Exemplos:\n{exemplos_invalidos}"
    )


dados["jogo_em_casa"] = eh_mandante.astype(bool)

dados["adversario_id"] = np.where(
    dados["jogo_em_casa"],
    dados["equipe_visitante_id"],
    dados["equipe_mandante_id"],
)


# ============================================================
# CHAVES TEMPORAIS
# ============================================================
#
# O histórico reinicia a cada temporada.
#
# Caso queira carregar o histórico entre temporadas, substitua:
#
#     ["ano", "atleta_id"]
#
# por:
#
#     ["atleta_id"]
# ============================================================

chaves_atleta_temporada = [
    "ano",
    "atleta_id",
]


# ============================================================
# FEATURES TEMPORAIS DOS SCOUTS
# ============================================================
#
# Todas as features usam somente partidas anteriores.
#
# Para os scouts:
# - *_ultima_1: valor na partida anterior
# - *_ultimas_3: soma nas 3 partidas anteriores
# - *_ultimas_5: soma nas 5 partidas anteriores
# - *_na_temporada: soma na temporada até a partida anterior
# ============================================================

for scout in scouts:
    grupo_scout = dados.groupby(
        chaves_atleta_temporada,
        sort=False,
        observed=True,
    )[scout]

    # Scout na partida imediatamente anterior
    dados[f"{scout}_ultima_1"] = (
        grupo_scout.shift(1)
    )

    # Total do scout nas últimas 3 partidas anteriores
    dados[f"{scout}_ultimas_3"] = (
        grupo_scout.transform(
            lambda serie: (
                serie
                .shift(1)
                .rolling(
                    window=3,
                    min_periods=1,
                )
                .sum()
            )
        )
    )

    # Total do scout nas últimas 5 partidas anteriores
    dados[f"{scout}_ultimas_5"] = (
        grupo_scout.transform(
            lambda serie: (
                serie
                .shift(1)
                .rolling(
                    window=5,
                    min_periods=1,
                )
                .sum()
            )
        )
    )

    # Total acumulado na temporada até a partida anterior
    dados[f"{scout}_na_temporada"] = (
        grupo_scout.cumsum()
        - dados[scout]
    )


# ============================================================
# FEATURES TEMPORAIS DE PREÇO
# ============================================================
#
# Para preço, usamos:
# - preço da partida anterior
# - média dos preços nas 3 partidas anteriores
# - média dos preços nas 5 partidas anteriores
# - média de preço na temporada até a partida anterior
#
# Não usamos soma de preços porque essa medida possui pouca
# interpretação econômica.
# ============================================================

grupo_preco = dados.groupby(
    chaves_atleta_temporada,
    sort=False,
    observed=True,
)["preco_num"]

dados["preco_num_ultima_1"] = (
    grupo_preco.shift(1)
)

dados["preco_num_ultimas_3"] = (
    grupo_preco.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .mean()
        )
    )
)

dados["preco_num_ultimas_5"] = (
    grupo_preco.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=5,
                min_periods=1,
            )
            .mean()
        )
    )
)

dados["preco_num_na_temporada"] = (
    grupo_preco.transform(
        lambda serie: (
            serie
            .shift(1)
            .expanding(
                min_periods=1,
            )
            .mean()
        )
    )
)


# Variação do preço entre as duas últimas observações anteriores.
dados["preco_num_variacao_ultima_1"] = (
    grupo_preco.shift(1)
    - grupo_preco.shift(2)
)


# ============================================================
# FEATURES TEMPORAIS DE PONTOS
# ============================================================
#
# Para pontos:
# - *_ultima_1: pontuação na partida anterior
# - *_ultimas_3: soma nas 3 partidas anteriores
# - *_ultimas_5: soma nas 5 partidas anteriores
# - *_na_temporada: soma na temporada até a partida anterior
#
# Também criamos as médias, pois são mais comparáveis entre
# jogadores com diferentes quantidades de partidas.
# ============================================================

grupo_pontos = dados.groupby(
    chaves_atleta_temporada,
    sort=False,
    observed=True,
)["pontos_num"]

dados["pontos_num_ultima_1"] = (
    grupo_pontos.shift(1)
)

dados["pontos_num_ultimas_3"] = (
    grupo_pontos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .sum()
        )
    )
)

dados["pontos_num_ultimas_5"] = (
    grupo_pontos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=5,
                min_periods=1,
            )
            .sum()
        )
    )
)

dados["pontos_num_na_temporada"] = (
    grupo_pontos.cumsum()
    - dados["pontos_num"].fillna(0)
)


dados["pontos_num_media_ultimas_3"] = (
    grupo_pontos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .mean()
        )
    )
)

dados["pontos_num_media_ultimas_5"] = (
    grupo_pontos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=5,
                min_periods=1,
            )
            .mean()
        )
    )
)

dados["pontos_num_media_na_temporada"] = (
    grupo_pontos.transform(
        lambda serie: (
            serie
            .shift(1)
            .expanding(
                min_periods=1,
            )
            .mean()
        )
    )
)


# ============================================================
# FEATURES TEMPORAIS DE JOGOS_NUM
# ============================================================
#
# jogos_num parece ser uma variável acumulada.
# Portanto, a principal feature é o valor disponível na
# observação anterior.
#
# As janelas de 3 e 5 abaixo usam o último valor disponível
# em cada janela, e não a soma de contagens acumuladas.
# ============================================================

grupo_jogos = dados.groupby(
    chaves_atleta_temporada,
    sort=False,
    observed=True,
)["jogos_num"]

dados["jogos_num_ultima_1"] = (
    grupo_jogos.shift(1)
)

dados["jogos_num_ultimas_3"] = (
    grupo_jogos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .max()
        )
    )
)

dados["jogos_num_ultimas_5"] = (
    grupo_jogos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=5,
                min_periods=1,
            )
            .max()
        )
    )
)

# Último valor acumulado disponível antes da partida.
dados["jogos_num_na_temporada"] = (
    grupo_jogos.shift(1)
)


# Diferença entre os dois últimos snapshots disponíveis.
dados["jogos_num_variacao_ultima_1"] = (
    grupo_jogos.shift(1)
    - grupo_jogos.shift(2)
)


# ============================================================
# MINUTOS HISTÓRICOS
# ============================================================

grupo_minutos = dados.groupby(
    chaves_atleta_temporada,
    sort=False,
    observed=True,
)["minutos_jogados"]

dados["minutos_ultima_1"] = (
    grupo_minutos.shift(1)
)

dados["minutos_ultimas_3"] = (
    grupo_minutos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .sum()
        )
    )
)

dados["minutos_ultimas_5"] = (
    grupo_minutos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=5,
                min_periods=1,
            )
            .sum()
        )
    )
)

dados["minutos_na_temporada"] = (
    grupo_minutos.cumsum()
    - dados["minutos_jogados"]
)


# Médias de minutos por partida nas janelas históricas.
dados["minutos_media_ultimas_3"] = (
    grupo_minutos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=3,
                min_periods=1,
            )
            .mean()
        )
    )
)

dados["minutos_media_ultimas_5"] = (
    grupo_minutos.transform(
        lambda serie: (
            serie
            .shift(1)
            .rolling(
                window=5,
                min_periods=1,
            )
            .mean()
        )
    )
)


# ============================================================
# SCOUTS POR 90 MINUTOS
# ============================================================
#
# Cria:
# - scout por 90 na partida anterior
# - scout por 90 nas últimas 3 partidas
# - scout por 90 nas últimas 5 partidas
# - scout por 90 acumulado na temporada
#
# Os denominadores usam somente minutos anteriores.
# ============================================================

for scout in scouts:
    dados[f"{scout}_por_90_ultima_1"] = np.where(
        dados["minutos_ultima_1"].gt(0),
        (
            90
            * dados[f"{scout}_ultima_1"]
            / dados["minutos_ultima_1"]
        ),
        np.nan,
    )

    dados[f"{scout}_por_90_ultimas_3"] = np.where(
        dados["minutos_ultimas_3"].gt(0),
        (
            90
            * dados[f"{scout}_ultimas_3"]
            / dados["minutos_ultimas_3"]
        ),
        np.nan,
    )

    dados[f"{scout}_por_90_ultimas_5"] = np.where(
        dados["minutos_ultimas_5"].gt(0),
        (
            90
            * dados[f"{scout}_ultimas_5"]
            / dados["minutos_ultimas_5"]
        ),
        np.nan,
    )

    dados[f"{scout}_por_90_na_temporada"] = np.where(
        dados["minutos_na_temporada"].gt(0),
        (
            90
            * dados[f"{scout}_na_temporada"]
            / dados["minutos_na_temporada"]
        ),
        np.nan,
    )


# ============================================================
# PONTOS POR 90 MINUTOS
# ============================================================

dados["pontos_num_por_90_ultima_1"] = np.where(
    dados["minutos_ultima_1"].gt(0),
    (
        90
        * dados["pontos_num_ultima_1"]
        / dados["minutos_ultima_1"]
    ),
    np.nan,
)

dados["pontos_num_por_90_ultimas_3"] = np.where(
    dados["minutos_ultimas_3"].gt(0),
    (
        90
        * dados["pontos_num_ultimas_3"]
        / dados["minutos_ultimas_3"]
    ),
    np.nan,
)

dados["pontos_num_por_90_ultimas_5"] = np.where(
    dados["minutos_ultimas_5"].gt(0),
    (
        90
        * dados["pontos_num_ultimas_5"]
        / dados["minutos_ultimas_5"]
    ),
    np.nan,
)

dados["pontos_num_por_90_na_temporada"] = np.where(
    dados["minutos_na_temporada"].gt(0),
    (
        90
        * dados["pontos_num_na_temporada"]
        / dados["minutos_na_temporada"]
    ),
    np.nan,
)


# ============================================================
# INDICADORES DE COBERTURA DO HISTÓRICO
# ============================================================

dados["partidas_anteriores_na_temporada"] = (
    dados.groupby(
        chaves_atleta_temporada,
        sort=False,
        observed=True,
    )
    .cumcount()
)

dados["sem_historico_na_temporada"] = (
    dados["partidas_anteriores_na_temporada"].eq(0)
)


dados["tem_3_partidas_anteriores"] = (
    dados["partidas_anteriores_na_temporada"].ge(3)
)

dados["tem_5_partidas_anteriores"] = (
    dados["partidas_anteriores_na_temporada"].ge(5)
)


# ============================================================
# IDENTIFICAÇÃO DOS TITULARES
# ============================================================

dados["eh_titular"] = (
    dados["status_pre_provável"].eq(1)
)

dados["eh_titular_defensivo"] = (
    dados["eh_titular"]
    & dados["posicao_id"].isin(posicoes_defensivas)
)

dados["eh_titular_ofensivo"] = (
    dados["eh_titular"]
    & dados["posicao_id"].isin(posicoes_ofensivas)
)


# Cria colunas auxiliares com o preço aplicável a cada grupo.
# Técnico não entra no valor dos atletas titulares.
dados["_preco_time_titular"] = (
    dados["preco_num_ultima_1"].where(
        dados["eh_titular"]
        & dados["posicao_id"].isin(posicoes_atletas)
    )
)

dados["_preco_defesa_titular"] = (
    dados["preco_num_ultima_1"].where(
        dados["eh_titular_defensivo"]
    )
)

dados["_preco_ataque_titular"] = (
    dados["preco_num_ultima_1"].where(
        dados["eh_titular_ofensivo"]
    )
)


# ============================================================
# VALOR AGREGADO DE CADA EQUIPE NA PARTIDA
# ============================================================

valores_equipes = (
    dados
    .groupby(
        [
            "ano",
            "match_id",
            "clube_id",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        valor_time_titular=(
            "_preco_time_titular",
            lambda serie: serie.sum(min_count=1),
        ),
        valor_parte_defensiva_titular=(
            "_preco_defesa_titular",
            lambda serie: serie.sum(min_count=1),
        ),
        valor_parte_ofensiva_titular=(
            "_preco_ataque_titular",
            lambda serie: serie.sum(min_count=1),
        ),
        quantidade_titulares=(
            "eh_titular",
            "sum",
        ),
        quantidade_titulares_defensivos=(
            "eh_titular_defensivo",
            "sum",
        ),
        quantidade_titulares_ofensivos=(
            "eh_titular_ofensivo",
            "sum",
        ),
    )
)


# ============================================================
# VALORES DO CLUBE DO ATLETA
# ============================================================

valores_clube = valores_equipes.rename(
    columns={
        "valor_time_titular":
            "clube_valor_time_titular",

        "valor_parte_defensiva_titular":
            "clube_valor_parte_defensiva_titular",

        "valor_parte_ofensiva_titular":
            "clube_valor_parte_ofensiva_titular",

        "quantidade_titulares":
            "clube_quantidade_titulares",

        "quantidade_titulares_defensivos":
            "clube_quantidade_titulares_defensivos",

        "quantidade_titulares_ofensivos":
            "clube_quantidade_titulares_ofensivos",
    }
)

dados = dados.merge(
    valores_clube,
    on=[
        "ano",
        "match_id",
        "clube_id",
    ],
    how="left",
    validate="many_to_one",
)


# ============================================================
# VALORES DO ADVERSÁRIO
# ============================================================

valores_adversario = valores_equipes.rename(
    columns={
        "clube_id":
            "adversario_id",

        "valor_time_titular":
            "adversario_valor_time_titular",

        "valor_parte_defensiva_titular":
            "adversario_valor_parte_defensiva_titular",

        "valor_parte_ofensiva_titular":
            "adversario_valor_parte_ofensiva_titular",

        "quantidade_titulares":
            "adversario_quantidade_titulares",

        "quantidade_titulares_defensivos":
            "adversario_quantidade_titulares_defensivos",

        "quantidade_titulares_ofensivos":
            "adversario_quantidade_titulares_ofensivos",
    }
)

dados = dados.merge(
    valores_adversario,
    on=[
        "ano",
        "match_id",
        "adversario_id",
    ],
    how="left",
    validate="many_to_one",
)


# ============================================================
# DIFERENÇAS DE VALOR ENTRE CLUBE E ADVERSÁRIO
# ============================================================

dados["diferenca_valor_time_titular"] = (
    dados["clube_valor_time_titular"]
    - dados["adversario_valor_time_titular"]
)

dados["diferenca_valor_parte_defensiva_titular"] = (
    dados["clube_valor_parte_defensiva_titular"]
    - dados["adversario_valor_parte_defensiva_titular"]
)

dados["diferenca_valor_parte_ofensiva_titular"] = (
    dados["clube_valor_parte_ofensiva_titular"]
    - dados["adversario_valor_parte_ofensiva_titular"]
)


# Ataque do clube comparado à defesa do adversário
dados["vantagem_valor_ataque_vs_defesa_adversaria"] = (
    dados["clube_valor_parte_ofensiva_titular"]
    - dados["adversario_valor_parte_defensiva_titular"]
)


# Defesa do clube comparada ao ataque do adversário
dados["vantagem_valor_defesa_vs_ataque_adversario"] = (
    dados["clube_valor_parte_defensiva_titular"]
    - dados["adversario_valor_parte_ofensiva_titular"]
)


# ============================================================
# REMOÇÃO DAS COLUNAS AUXILIARES
# ============================================================

dados = dados.drop(
    columns=[
        "_preco_time_titular",
        "_preco_defesa_titular",
        "_preco_ataque_titular",
    ]
)


# ============================================================
# VALIDAÇÕES FINAIS
# ============================================================

# Cada partida deve possuir exatamente duas equipes.
quantidade_equipes_por_partida = (
    dados.groupby(
        [
            "ano",
            "match_id",
        ],
        observed=True,
    )["clube_id"]
    .nunique()
)

partidas_com_quantidade_invalida = (
    quantidade_equipes_por_partida.loc[
        quantidade_equipes_por_partida.ne(2)
    ]
)

if not partidas_com_quantidade_invalida.empty:
    print(
        "Aviso: há partidas que não possuem exatamente duas "
        "equipes representadas no DataFrame."
    )

    display(
        partidas_com_quantidade_invalida.head(10)
    )


# Verifica a quantidade de titulares identificados por equipe.
equipes_com_titulares_incompletos = (
    valores_equipes.loc[
        valores_equipes["quantidade_titulares"].ne(11),
        [
            "ano",
            "match_id",
            "clube_id",
            "quantidade_titulares",
            "quantidade_titulares_defensivos",
            "quantidade_titulares_ofensivos",
        ],
    ]
)

if not equipes_com_titulares_incompletos.empty:
    print(
        "Aviso: há equipes com quantidade de titulares "
        "diferente de 11. Confira a cobertura da base."
    )

    print(
        "Quantidade de casos:",
        equipes_com_titulares_incompletos.shape[0],
    )

    display(
        equipes_com_titulares_incompletos.head(10)
    )


# ============================================================
# VALIDAÇÃO DE VAZAMENTO TEMPORAL
# ============================================================
#
# Nesta etapa, as variáveis da partida atual continuam no
# DataFrame porque podem ser necessárias como alvo ou auditoria.
#
# Para modelagem, não inclua:
# - pontos_num atual
# - minutos_jogados atual
# - scouts atuais
#
# Utilize somente as respectivas versões históricas.
# ============================================================

colunas_proibidas_como_features = [
    *scouts,
    "pontos_num",
    "minutos_jogados",
]

print(
    "Não utilizar como features da partida atual:",
    colunas_proibidas_como_features,
)


# ============================================================
# VISUALIZAÇÃO DO RESULTADO
# ============================================================

colunas_exemplo = [
    "ano",
    "rodada_id",
    "rodada_id_corrigido",
    "match_id",
    "atleta_id",
    "clube_id",
    "adversario_id",
    "jogo_em_casa",

    "G",
    "G_ultima_1",
    "G_ultimas_3",
    "G_ultimas_5",
    "G_na_temporada",
    "G_por_90_ultima_1",
    "G_por_90_ultimas_3",
    "G_por_90_ultimas_5",
    "G_por_90_na_temporada",

    "preco_num",
    "preco_num_ultima_1",
    "preco_num_ultimas_3",
    "preco_num_ultimas_5",
    "preco_num_na_temporada",
    "preco_num_variacao_ultima_1",

    "jogos_num",
    "jogos_num_ultima_1",
    "jogos_num_ultimas_3",
    "jogos_num_ultimas_5",
    "jogos_num_na_temporada",

    "pontos_num",
    "pontos_num_ultima_1",
    "pontos_num_ultimas_3",
    "pontos_num_ultimas_5",
    "pontos_num_na_temporada",
    "pontos_num_media_ultimas_3",
    "pontos_num_media_ultimas_5",
    "pontos_num_media_na_temporada",
    "pontos_num_por_90_ultimas_3",
    "pontos_num_por_90_ultimas_5",
    "pontos_num_por_90_na_temporada",

    "minutos_ultima_1",
    "minutos_ultimas_3",
    "minutos_ultimas_5",
    "minutos_na_temporada",

    "partidas_anteriores_na_temporada",
    "sem_historico_na_temporada",
    "tem_3_partidas_anteriores",
    "tem_5_partidas_anteriores",

    "clube_valor_time_titular",
    "clube_valor_parte_defensiva_titular",
    "clube_valor_parte_ofensiva_titular",

    "adversario_valor_time_titular",
    "adversario_valor_parte_defensiva_titular",
    "adversario_valor_parte_ofensiva_titular",

    "diferenca_valor_time_titular",
    "vantagem_valor_ataque_vs_defesa_adversaria",
    "vantagem_valor_defesa_vs_ataque_adversario",
]

display(
    dados[colunas_exemplo].head(20)
)

/var/folders/yb/53f7g_5x5sj34hp_4zc0xj580000gn/T/ipykernel_94575/105334599.py:309: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[f"{scout}_na_temporada"] = (
/var/folders/yb/53f7g_5x5sj34hp_4zc0xj580000gn/T/ipykernel_94575/105334599.py:274: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[f"{scout}_ultima_1"] = (
/var/folders/yb/53f7g_5x5sj34hp_4zc0xj580000gn/T/ipykernel_94575/105334599.py:279: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

Aviso: há equipes com quantidade de titulares diferente de 11. Confira a cobertura da base.
Quantidade de casos: 2370


/var/folders/yb/53f7g_5x5sj34hp_4zc0xj580000gn/T/ipykernel_94575/105334599.py:614: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados["minutos_media_ultimas_5"] = (
/var/folders/yb/53f7g_5x5sj34hp_4zc0xj580000gn/T/ipykernel_94575/105334599.py:643: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dados[f"{scout}_por_90_ultima_1"] = np.where(
/var/folders/yb/53f7g_5x5sj34hp_4zc0xj580000gn/T/ipykernel_94575/105334599.py:653: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

,ano,match_id,clube_id,quantidade_titulares,quantidade_titulares_defensivos,quantidade_titulares_ofensivos
1,2022,900001,114,12,5,6
2,2022,900002,105,12,6,5
3,2022,900002,109,12,5,6
4,2022,900003,108,12,5,6
7,2022,900004,124,12,5,6
8,2022,900005,102,12,5,6
10,2022,900006,110,12,5,6
12,2022,900007,125,12,4,7
13,2022,900007,127,12,5,6
15,2022,900008,126,12,4,7


Não utilizar como features da partida atual: ['G', 'A', 'SG', 'FF', 'FT', 'FD', 'DP', 'DE', 'DS', 'FS', 'PS', 'GS', 'GC', 'CA', 'CV', 'FC', 'I', 'PP', 'PC', 'pontos_num', 'minutos_jogados']


,ano,rodada_id,rodada_id_corrigido,match_id,atleta_id,clube_id,adversario_id,jogo_em_casa,G,G_ultima_1,G_ultimas_3,G_ultimas_5,G_na_temporada,G_por_90_ultima_1,G_por_90_ultimas_3,G_por_90_ultimas_5,G_por_90_na_temporada,preco_num,preco_num_ultima_1,preco_num_ultimas_3,preco_num_ultimas_5,preco_num_na_temporada,preco_num_variacao_ultima_1,jogos_num,jogos_num_ultima_1,jogos_num_ultimas_3,jogos_num_ultimas_5,jogos_num_na_temporada,pontos_num,pontos_num_ultima_1,pontos_num_ultimas_3,pontos_num_ultimas_5,pontos_num_na_temporada,pontos_num_media_ultimas_3,pontos_num_media_ultimas_5,pontos_num_media_na_temporada,pontos_num_por_90_ultimas_3,pontos_num_por_90_ultimas_5,pontos_num_por_90_na_temporada,minutos_ultima_1,minutos_ultimas_3,minutos_ultimas_5,minutos_na_temporada,partidas_anteriores_na_temporada,sem_historico_na_temporada,tem_3_partidas_anteriores,tem_5_partidas_anteriores,clube_valor_time_titular,clube_valor_parte_defensiva_titular,clube_valor_parte_ofensiva_titular,adversario_valor_time_titular,adversario_valor_parte_defensiva_titular,adversario_valor_parte_ofensiva_titular,diferenca_valor_time_titular,vantagem_valor_ataque_vs_defesa_adversaria,vantagem_valor_defesa_vs_ataque_adversario
0,2022,1,0,900001,10028,111,114.0,True,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,7.00,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,1,0,900001,10032,111,114.0,True,2.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,23.05,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,18.90,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,1,0,900001,10065,111,114.0,True,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,6.00,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,1,0,900001,10084,114,111.0,False,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,5.00,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,1,0,900001,10087,114,111.0,False,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,6.08,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,0.20,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2022,1,0,900001,10102,114,111.0,False,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,12.88,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,7.50,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2022,1,0,900001,10103,114,111.0,False,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,5.00,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2022,1,0,900001,10104,114,111.0,False,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,7.00,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2022,1,0,900001,10108,111,114.0,True,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,6.00,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2022,1,0,900001,10125,111,114.0,True,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,5.00,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
backup = dados.copy()

In [14]:
dados = backup.copy()

In [15]:
utils.data_exploratory_analysis(dados)

===== Data Inspection =====

Shape:
(109952, 254)

Data Types:
atleta_id                                       int64
ano                                             int64
rodada_id                                       int64
clube_id                                        int64
posicao_id                                      int64
preco_num                                     float64
variacao_num                                  float64
media_num                                     float64
jogos_num                                       int64
pontos_num                                    float64
minutos_jogados                               float64
entrou_em_campo                                  bool
match_id                                        int64
G                                             float64
A                                             float64
SG                                            float64
FF                                            float64
FT                 

In [16]:
colunas = [
    'atleta_id',
    'clube_id',
    'ano',
    'rodada_id',
    "rodada_id_corrigido",
    'adversario_id',
    'adversario_media_pontos_cedidos',
    'adversario_media_pontos_cedidos_ausencia',
    # 'adversario_quantidade_titulares',
    # 'adversario_quantidade_titulares_defensivos',
    # 'adversario_quantidade_titulares_ofensivos',
    'adversario_valor_parte_defensiva_titular',
    'adversario_valor_parte_ofensiva_titular',
    'adversario_valor_time_titular',
    #  'clube_quantidade_titulares',
    #  'clube_quantidade_titulares_defensivos',
    #  'clube_quantidade_titulares_ofensivos',
    'clube_valor_parte_defensiva_titular',
    'clube_valor_parte_ofensiva_titular',
    'clube_valor_time_titular',
    'diferenca_valor_parte_defensiva_titular',
    'diferenca_valor_parte_ofensiva_titular',
    'diferenca_valor_time_titular',
    #  'eh_titular',
    #  'eh_titular_defensivo',
    #  'eh_titular_ofensivo',
    'entrou_em_campo',
    #  'equipe_mandante_id',
    'equipe_media_pontos_conquistados',
    'equipe_media_pontos_conquistados_ausencia',
    'equipe_visitante_id',
    'jogo_em_casa',
    # 'jogos_num',
    'jogos_num_na_temporada',
    'jogos_num_ultima_1',
    'jogos_num_ultimas_3',
    'jogos_num_ultimas_5',
    'jogos_num_variacao_ultima_1',
    #  'match_id',
    # 'media_num',
    #  'minutos_jogados',
    #  'minutos_jogados_ausencia',
    'minutos_media_ultimas_3',
    'minutos_media_ultimas_5',
    'minutos_na_temporada',
    'minutos_ultima_1',
    'minutos_ultimas_3',
    'minutos_ultimas_5',
    #  'momento_entrou',
    #  'momento_entrou_ausencia',
    #  'momento_substituido',
    #  'momento_substituido_ausencia',
    # 'partidas_anteriores_na_temporada',
    'pontos_num',
    'pontos_num_media_na_temporada',
    'pontos_num_media_ultimas_3',
    'pontos_num_media_ultimas_5',
    'pontos_num_na_temporada',
    'pontos_num_por_90_na_temporada',
    'pontos_num_por_90_ultima_1',
    'pontos_num_por_90_ultimas_3',
    'pontos_num_por_90_ultimas_5',
    'pontos_num_ultima_1',
    'pontos_num_ultimas_3',
    'pontos_num_ultimas_5',
    'posicao_id',
    # 'preco_num',
    'preco_num_na_temporada',
    'preco_num_ultima_1',
    'preco_num_ultimas_3',
    'preco_num_ultimas_5',
    #  'preco_num_variacao_ultima_1',
    'sem_historico_na_temporada',
    'status_pre_contundido',
    'status_pre_dúvida',
    'status_pre_nulo',
    'status_pre_provável',
    'status_pre_suspenso',
    #  'status_inicial_nao_relacionado',
    #  'status_inicial_reserva',
    #  'status_pre_provável',
    #  'tem_3_partidas_anteriores',
    #  'tem_5_partidas_anteriores',
    'vantagem_valor_ataque_vs_defesa_adversaria',
    'vantagem_valor_defesa_vs_ataque_adversario',
    #  'variacao_num',
    'A',
    'A_na_temporada',
    'A_por_90_na_temporada',
    'A_por_90_ultima_1',
    'A_por_90_ultimas_3',
    'A_por_90_ultimas_5',
    'A_ultima_1',
    'A_ultimas_3',
    'A_ultimas_5',
    'CA',
    'CA_na_temporada',
    'CA_por_90_na_temporada',
    'CA_por_90_ultima_1',
    'CA_por_90_ultimas_3',
    'CA_por_90_ultimas_5',
    'CA_ultima_1',
    'CA_ultimas_3',
    'CA_ultimas_5',
    'CV',
    'CV_na_temporada',
    'CV_por_90_na_temporada',
    'CV_por_90_ultima_1',
    'CV_por_90_ultimas_3',
    'CV_por_90_ultimas_5',
    'CV_ultima_1',
    'CV_ultimas_3',
    'CV_ultimas_5',
    'DE',
    'DE_na_temporada',
    'DE_por_90_na_temporada',
    'DE_por_90_ultima_1',
    'DE_por_90_ultimas_3',
    'DE_por_90_ultimas_5',
    'DE_ultima_1',
    'DE_ultimas_3',
    'DE_ultimas_5',
    'DP',
    'DP_na_temporada',
    'DP_por_90_na_temporada',
    'DP_por_90_ultima_1',
    'DP_por_90_ultimas_3',
    'DP_por_90_ultimas_5',
    'DP_ultima_1',
    'DP_ultimas_3',
    'DP_ultimas_5',
    'DS',
    'DS_na_temporada',
    'DS_por_90_na_temporada',
    'DS_por_90_ultima_1',
    'DS_por_90_ultimas_3',
    'DS_por_90_ultimas_5',
    'DS_ultima_1',
    'DS_ultimas_3',
    'DS_ultimas_5',
    'FC',
    'FC_na_temporada',
    'FC_por_90_na_temporada',
    'FC_por_90_ultima_1',
    'FC_por_90_ultimas_3',
    'FC_por_90_ultimas_5',
    'FC_ultima_1',
    'FC_ultimas_3',
    'FC_ultimas_5',
    'FD',
    'FD_na_temporada',
    'FD_por_90_na_temporada',
    'FD_por_90_ultima_1',
    'FD_por_90_ultimas_3',
    'FD_por_90_ultimas_5',
    'FD_ultima_1',
    'FD_ultimas_3',
    'FD_ultimas_5',
    'FF',
    'FF_na_temporada',
    'FF_por_90_na_temporada',
    'FF_por_90_ultima_1',
    'FF_por_90_ultimas_3',
    'FF_por_90_ultimas_5',
    'FF_ultima_1',
    'FF_ultimas_3',
    'FF_ultimas_5',
    'FS',
    'FS_na_temporada',
    'FS_por_90_na_temporada',
    'FS_por_90_ultima_1',
    'FS_por_90_ultimas_3',
    'FS_por_90_ultimas_5',
    'FS_ultima_1',
    'FS_ultimas_3',
    'FS_ultimas_5',
    'FT',
    'FT_na_temporada',
    'FT_por_90_na_temporada',
    'FT_por_90_ultima_1',
    'FT_por_90_ultimas_3',
    'FT_por_90_ultimas_5',
    'FT_ultima_1',
    'FT_ultimas_3',
    'FT_ultimas_5',
    'G',
    'G_na_temporada',
    'G_por_90_na_temporada',
    'G_por_90_ultima_1',
    'G_por_90_ultimas_3',
    'G_por_90_ultimas_5',
    'G_ultima_1',
    'G_ultimas_3',
    'G_ultimas_5',
    'GC',
    'GC_na_temporada',
    'GC_por_90_na_temporada',
    'GC_por_90_ultima_1',
    'GC_por_90_ultimas_3',
    'GC_por_90_ultimas_5',
    'GC_ultima_1',
    'GC_ultimas_3',
    'GC_ultimas_5',
    'GS',
    'GS_na_temporada',
    'GS_por_90_na_temporada',
    'GS_por_90_ultima_1',
    'GS_por_90_ultimas_3',
    'GS_por_90_ultimas_5',
    'GS_ultima_1',
    'GS_ultimas_3',
    'GS_ultimas_5',
    'I',
    'I_na_temporada',
    'I_por_90_na_temporada',
    'I_por_90_ultima_1',
    'I_por_90_ultimas_3',
    'I_por_90_ultimas_5',
    'I_ultima_1',
    'I_ultimas_3',
    'I_ultimas_5',
    'PC',
    'PC_na_temporada',
    'PC_por_90_na_temporada',
    'PC_por_90_ultima_1',
    'PC_por_90_ultimas_3',
    'PC_por_90_ultimas_5',
    'PC_ultima_1',
    'PC_ultimas_3',
    'PC_ultimas_5',
    'PP',
    'PP_na_temporada',
    'PP_por_90_na_temporada',
    'PP_por_90_ultima_1',
    'PP_por_90_ultimas_3',
    'PP_por_90_ultimas_5',
    'PP_ultima_1',
    'PP_ultimas_3',
    'PP_ultimas_5',
    'PS',
    'PS_na_temporada',
    'PS_por_90_na_temporada',
    'PS_por_90_ultima_1',
    'PS_por_90_ultimas_3',
    'PS_por_90_ultimas_5',
    'PS_ultima_1',
    'PS_ultimas_3',
    'PS_ultimas_5',
    'SG',
    'SG_na_temporada',
    'SG_por_90_na_temporada',
    'SG_por_90_ultima_1',
    'SG_por_90_ultimas_3',
    'SG_por_90_ultimas_5',
    'SG_ultima_1',
    'SG_ultimas_3',
    'SG_ultimas_5',
]

In [17]:
dados = dados.fillna(0)

# Exportação dos dados tratados

In [18]:
export = dados[colunas].copy()

caminho = Path('../outputs/base_gm_tratada.parquet')
export.to_parquet(caminho)